# Clase 8 — Optimización de Rutas y Operaciones de Flota Pesquera

**Curso:** Inteligencia Artificial Aplicada a la Producción Pesquera  
**Institución:** UTN Facultad Regional Chubut | PesquerosEnIA  
**Autor:** Ariel Giamportone  
**Fecha:** 2026  
**Licencia:** GPL-3.0

---

**Objetivo:** Aplicar técnicas de optimización y machine learning para mejorar la eficiencia  
operativa de embarcaciones pesqueras: minimizar consumo de combustible, optimizar velocidad  
de crucero, planificar campañas y anticipar necesidades de mantenimiento.

**Contenido:**
1. Setup
2. Contexto: el costo del combustible en la flota pesquera argentina
3. Parte A: Modelo predictivo de consumo de combustible
4. Parte B: Optimización de velocidad de crucero
5. Parte C: Clustering de zonas para planificación de campaña
6. Parte D: Mantenimiento predictivo de motores (introducción)
7. Simulación: impacto económico de la optimización
8. Reflexión y ética

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, classification_report
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (11, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
np.random.seed(42)
print('✓ Librerías cargadas')

## 2. Contexto: el combustible es el mayor costo operativo

En la industria pesquera argentina de altura, el combustible (gasoil / fuel oil) representa  
entre el **25% y el 40% de los costos operativos totales** de una marea.

**Datos de referencia (buque arrastrero de altura típico):**
- Consumo promedio: 15–25 toneladas de gasoil por marea
- Precio gasoil (ref. 2025): ~USD 800–900/tn
- Costo de combustible por marea: USD 12.000–22.000
- Mareas por año: 10–14

**Potencial de ahorro con optimización:** estudios internacionales demuestran reducciones  
del 10–20% con optimización de velocidad y rutas. Para una flota de 20 arrastreros,  
eso representa **USD 800.000–2.000.000/año** en ahorro directo.

## Parte A: Modelo predictivo de consumo de combustible

Construimos un modelo que predice el consumo de combustible de una marea  
a partir de sus características operativas y ambientales. Este modelo permite:
- Presupuestar mejor el costo de cada viaje
- Identificar qué factores aumentan el consumo innecesariamente
- Comparar la eficiencia entre barcos de la misma flota

In [ ]:
# ── Dataset histórico de mareas: variables operativas ─────────────────────────
np.random.seed(42)
n_mareas = 500

datos_operativos = pd.DataFrame({
    # Variables del viaje
    'distancia_puerto_km': np.random.uniform(100, 600, n_mareas),
    'dias_en_mar':         np.random.uniform(5, 20, n_mareas),
    'velocidad_media_kn':  np.random.normal(9.0, 1.5, n_mareas),

    # Condiciones ambientales
    'viento_medio_kn':     np.abs(np.random.normal(15, 8, n_mareas)),
    'estado_mar':          np.random.choice([1, 2, 3, 4], n_mareas, p=[0.20, 0.40, 0.30, 0.10]),

    # Operación de pesca
    'horas_arrastre':      np.random.uniform(40, 140, n_mareas),
    'profundidad_pesca_m': np.random.uniform(80, 350, n_mareas),
    'toneladas_captura':   np.abs(np.random.normal(80, 30, n_mareas)),

    # Estado del barco
    'antiguedad_motor_anos':  np.random.uniform(1, 15, n_mareas),
    'mantenimiento_reciente': np.random.choice([0, 1], n_mareas, p=[0.60, 0.40])
})

# Velocidad clipeada (realista)
datos_operativos['velocidad_media_kn'] = datos_operativos['velocidad_media_kn'].clip(5, 13)

# ── Consumo de combustible (variable objetivo) ────────────────────────────────
consumo_base = (
    datos_operativos['distancia_puerto_km'] * 0.08 +
    datos_operativos['dias_en_mar'] * 3.5 +
    datos_operativos['velocidad_media_kn'] ** 2 * 0.15 +
    datos_operativos['viento_medio_kn'] * 0.30 +
    datos_operativos['estado_mar'] * 4.0 +
    datos_operativos['horas_arrastre'] * 0.50 +
    datos_operativos['profundidad_pesca_m'] * 0.02 +
    datos_operativos['antiguedad_motor_anos'] * 1.2 -
    datos_operativos['mantenimiento_reciente'] * 5.0
)
datos_operativos['consumo_combustible_tn'] = (
    consumo_base + np.random.normal(0, 5, n_mareas)
).clip(20, 180)

print(f'Dataset: {datos_operativos.shape}')
print(f'Consumo promedio: {datos_operativos["consumo_combustible_tn"].mean():.1f} tn/marea')
print(f'Consumo mínimo: {datos_operativos["consumo_combustible_tn"].min():.1f} tn/marea')
print(f'Consumo máximo: {datos_operativos["consumo_combustible_tn"].max():.1f} tn/marea')

In [ ]:
# ── EDA: distribución y correlaciones del consumo ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma
axes[0].hist(datos_operativos['consumo_combustible_tn'], bins=30,
             color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(datos_operativos['consumo_combustible_tn'].mean(),
                color='red', linestyle='--', linewidth=2,
                label=f'Media: {datos_operativos["consumo_combustible_tn"].mean():.0f} tn')
axes[0].set_xlabel('Consumo de combustible (tn/marea)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución del consumo de combustible')
axes[0].legend()

# Correlación con consumo
features_corr = ['distancia_puerto_km', 'dias_en_mar', 'velocidad_media_kn',
                 'viento_medio_kn', 'estado_mar', 'horas_arrastre']
corr_consumo = datos_operativos[features_corr + ['consumo_combustible_tn']].corr()[
    'consumo_combustible_tn'].drop('consumo_combustible_tn').sort_values()

colores_corr = ['#E74C3C' if v > 0 else '#3498DB' for v in corr_consumo]
corr_consumo.plot(kind='barh', ax=axes[1], color=colores_corr, alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Correlación con consumo de combustible')
axes[1].set_title('¿Qué factores más afectan el consumo?')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('Análisis del Consumo de Combustible — Flota Pesquera PCA', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Modelo predictivo de consumo (Random Forest Regressor) ────────────────────
features_consumo = ['distancia_puerto_km', 'dias_en_mar', 'velocidad_media_kn',
                    'viento_medio_kn', 'estado_mar', 'horas_arrastre',
                    'profundidad_pesca_m', 'antiguedad_motor_anos', 'mantenimiento_reciente']

X_c = datos_operativos[features_consumo]
y_c = datos_operativos['consumo_combustible_tn']

X_train, X_test, y_train, y_test = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

modelo_consumo = RandomForestRegressor(n_estimators=100, random_state=42)
modelo_consumo.fit(X_train, y_train)
y_pred_c = modelo_consumo.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_c)
r2 = r2_score(y_test, y_pred_c)
print(f'Error Absoluto Medio (MAE): {mae:.1f} toneladas de gasoil')
print(f'R² (coef. determinación): {r2:.3f}')
print(f'Traducción: el modelo predice el consumo con un error promedio de {mae:.0f} tn')
print(f'  ≈ ${mae * 850:,.0f} USD de error en la estimación de costo por marea')

# Scatter real vs predicho
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test, y_pred_c, alpha=0.4, s=25, color='steelblue')
lims = [min(y_test.min(), y_pred_c.min()), max(y_test.max(), y_pred_c.max())]
ax.plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Consumo real (tn)')
ax.set_ylabel('Consumo predicho (tn)')
ax.set_title(f'Predicción de consumo de combustible\nMAE={mae:.1f} tn | R²={r2:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

## Parte B: Optimización de velocidad de crucero

La velocidad de un barco sigue la **ley cúbica del consumo naval**: duplicar la velocidad
multiplica el consumo por 8 aproximadamente. Pero ir muy lento también es costoso
(más días en el mar = más costo de tripulación y operación).

Existe una **velocidad óptima** que minimiza el costo total por viaje.

In [ ]:
# ── Curva de costo total vs velocidad ─────────────────────────────────────────
velocidades = np.arange(5.5, 13.5, 0.1)  # nudos
distancia_caladero_km = 300  # km al caladero (ejemplo: Puerto Madryn → zona merluza)

# Conversión: 1 nudo = 1.852 km/h
velocidades_kmh = velocidades * 1.852
tiempo_viaje_h = distancia_caladero_km / velocidades_kmh  # horas de navegación

# Consumo: función cúbica de velocidad (ley de Admiralty)
# C(v) = k * v^3 * tiempo (simplificado)
k_consumo = 0.003  # constante para buque tipo arrastrero
consumo_viaje_tn = k_consumo * (velocidades ** 3) * (distancia_caladero_km / velocidades_kmh)

# Costo de combustible
precio_gasoil_usd_tn = 850
costo_combustible = consumo_viaje_tn * precio_gasoil_usd_tn

# Costo operativo del tiempo (tripulación + charter + costos fijos)
costo_por_hora_usd = 1_400  # USD/hora en navegación
costo_tiempo = tiempo_viaje_h * costo_por_hora_usd

# Costo total
costo_total = costo_combustible + costo_tiempo

velocidad_optima = velocidades[np.argmin(costo_total)]
print(f'Velocidad de mínimo consumo: {velocidades[np.argmin(consumo_viaje_tn)]:.1f} kn')
print(f'Velocidad de mínimo costo total: {velocidad_optima:.1f} kn')
print(f'Costo total mínimo por viaje (ida al caladero): USD {min(costo_total):,.0f}')

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(velocidades, consumo_viaje_tn, 'steelblue', linewidth=2)
axes[0].axvline(velocidades[np.argmin(consumo_viaje_tn)], linestyle='--',
                color='gray', label=f'Mín. consumo: {velocidades[np.argmin(consumo_viaje_tn)]:.1f} kn')
axes[0].set_xlabel('Velocidad (nudos)')
axes[0].set_ylabel('Consumo de combustible (tn)')
axes[0].set_title('Consumo vs Velocidad (dist. 300 km)\nLey cúbica del consumo naval')
axes[0].legend()

axes[1].plot(velocidades, costo_total / 1000, 'darkorange', linewidth=2, label='Costo total')
axes[1].plot(velocidades, costo_combustible / 1000, 'steelblue', linestyle='--',
             linewidth=1.5, alpha=0.7, label='Solo combustible')
axes[1].plot(velocidades, costo_tiempo / 1000, 'green', linestyle='--',
             linewidth=1.5, alpha=0.7, label='Solo tiempo (tripulación)')
axes[1].axvline(velocidad_optima, linestyle='-', color='red', linewidth=2,
                label=f'Velocidad óptima: {velocidad_optima:.1f} kn')
axes[1].set_xlabel('Velocidad (nudos)')
axes[1].set_ylabel('Costo (miles de USD)')
axes[1].set_title('Costo total vs Velocidad\n(combustible + tiempo operativo)')
axes[1].legend(fontsize=9)

plt.suptitle('Optimización de Velocidad de Crucero — Arrastrero PCA (dist. 300 km)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Parte C: Clustering de zonas para planificación de campaña

El algoritmo **K-Means** agrupa las zonas de pesca históricas según su eficiencia  
(captura por unidad de consumo). Esto permite identificar los "clusters" de zonas  
de alta, media y baja eficiencia para planificar la próxima campaña pesquera.

In [ ]:
# ── Dataset: historial de zonas de pesca con eficiencia ───────────────────────
np.random.seed(42)
n_zonas = 250

historial_zonas = pd.DataFrame({
    'latitud':              np.random.uniform(-52, -38, n_zonas),
    'longitud':             np.random.uniform(-62, -48, n_zonas),
    'captura_toneladas':    np.abs(np.random.normal(75, 28, n_zonas)),
    'consumo_relativo':     np.clip(np.random.normal(1.0, 0.2, n_zonas), 0.5, 1.8),
    'mes':                  np.random.randint(1, 13, n_zonas)
})

# Eficiencia: toneladas capturadas por unidad de combustible consumido
historial_zonas['eficiencia'] = (
    historial_zonas['captura_toneladas'] / historial_zonas['consumo_relativo']
)

# K-Means con 4 clusters
X_kmeans = historial_zonas[['latitud', 'longitud', 'eficiencia']].copy()
scaler_km = StandardScaler()
X_kmeans_s = scaler_km.fit_transform(X_kmeans)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
historial_zonas['cluster'] = kmeans.fit_predict(X_kmeans_s)

# Resumen por cluster
resumen_clusters = historial_zonas.groupby('cluster').agg({
    'captura_toneladas': 'mean',
    'consumo_relativo': 'mean',
    'eficiencia': 'mean',
    'latitud': 'mean'
}).round(2)
resumen_clusters.index = ['Cluster A', 'Cluster B', 'Cluster C', 'Cluster D']
print('Resumen de clusters de zonas de pesca:')
print(resumen_clusters)

In [ ]:
# ── Visualización de clusters ──────────────────────────────────────────────────
colores_cluster = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']
labels_cluster = ['Zona A', 'Zona B', 'Zona C', 'Zona D']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Mapa de clusters
for i, (color, label) in enumerate(zip(colores_cluster, labels_cluster)):
    mask = historial_zonas['cluster'] == i
    axes[0].scatter(
        historial_zonas[mask]['longitud'], historial_zonas[mask]['latitud'],
        c=color, label=f'{label} (n={mask.sum()})', alpha=0.6, s=40
    )

axes[0].set_xlabel('Longitud (°O)')
axes[0].set_ylabel('Latitud (°S)')
axes[0].set_title('Clusters de Zonas de Pesca — PCA\n(agrupadas por eficiencia operativa)',
                  fontsize=11)
axes[0].legend(fontsize=9)

# Eficiencia promedio por cluster
eficiencia_media = historial_zonas.groupby('cluster')['eficiencia'].mean()
axes[1].bar(labels_cluster, eficiencia_media.values,
            color=colores_cluster, alpha=0.85, edgecolor='white')

# Marcar el mejor cluster
mejor_cluster_idx = eficiencia_media.idxmax()
axes[1].bar(labels_cluster[mejor_cluster_idx], eficiencia_media.iloc[mejor_cluster_idx],
            color=colores_cluster[mejor_cluster_idx], edgecolor='gold', linewidth=3,
            label='★ Zona más eficiente')
axes[1].set_ylabel('Eficiencia media (tn captura / unidad combustible)')
axes[1].set_title('Eficiencia por Cluster de Zona\n(mayor = más captura por litro de gasoil)',
                  fontsize=11)
axes[1].legend(fontsize=10)

plt.suptitle('Análisis de Eficiencia por Zona de Pesca — K-Means (k=4)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nZona de mayor eficiencia promedio: Cluster {labels_cluster[mejor_cluster_idx]}')
print(f'Recomendación: priorizar esta zona en la próxima campaña')

## Parte D: Mantenimiento predictivo — introducción

Un motor de barco que falla en alta mar puede costar:
- Costo de reparación de emergencia: USD 50.000–200.000
- Pérdida de marea: USD 150.000–400.000
- Riesgo de seguridad para la tripulación

Un modelo de **mantenimiento predictivo** analiza las señales del motor (temperatura,
presión de aceite, vibración) para detectar anomalías antes de que ocurra la falla.

In [ ]:
# ── Dataset de telemetría de motor ────────────────────────────────────────────
np.random.seed(42)
n_registros = 400

datos_motor = pd.DataFrame({
    'horas_operacion':          np.random.uniform(200, 8000, n_registros),
    'temperatura_motor_c':      np.random.normal(82, 8, n_registros),
    'presion_aceite_bar':        np.random.normal(4.2, 0.5, n_registros),
    'vibracion_mm_s':           np.abs(np.random.normal(2.8, 1.2, n_registros)),
    'consumo_aceite_l_100h':    np.random.normal(2.5, 0.7, n_registros),
    'dias_desde_ultimo_service': np.random.uniform(0, 180, n_registros)
})

# Falla inminente: cuando varias señales están fuera de rango simultáneamente
falla = (
    (datos_motor['temperatura_motor_c'] > 96) |
    (datos_motor['presion_aceite_bar'] < 3.1) |
    (datos_motor['vibracion_mm_s'] > 5.8) |
    ((datos_motor['horas_operacion'] > 6000) & (datos_motor['dias_desde_ultimo_service'] > 120))
).astype(int)
datos_motor['falla_proxima'] = falla

print(f'Registros de telemetría: {len(datos_motor)}')
print(f'Alertas de falla detectadas: {falla.sum()} ({falla.mean():.1%})')

# Modelo de clasificación
X_m = datos_motor.drop('falla_proxima', axis=1)
y_m = datos_motor['falla_proxima']

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_m, y_m, test_size=0.2, random_state=42, stratify=y_m
)
modelo_mant = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_mant.fit(X_m_train, y_m_train)
y_m_pred = modelo_mant.predict(X_m_test)

print('\nReporte del modelo de mantenimiento predictivo:')
print(classification_report(y_m_test, y_m_pred, target_names=['Normal', 'Falla inminente']))

## 7. Simulación: impacto económico de la optimización

Comparamos el consumo de una flota de 20 arrastreros de altura **sin** y **con**  
optimización por IA a lo largo de un año (12 mareas por barco).

In [ ]:
# ── Simulación de impacto económico ───────────────────────────────────────────
np.random.seed(42)
n_barcos = 20
n_mareas_anio = 12
precio_gasoil = 850  # USD/tn

# Consumo sin optimización: promedio ~90 tn/marea, con variabilidad
consumo_sin_opt = np.random.normal(92, 12, (n_barcos, n_mareas_anio)).clip(50, 160)

# Con optimización: reducción del 12-18% (velocidad + rutas + mantenimiento)
factor_reduccion = np.random.uniform(0.82, 0.88, (n_barcos, n_mareas_anio))
consumo_con_opt = consumo_sin_opt * factor_reduccion

ahorro_tn = consumo_sin_opt - consumo_con_opt
ahorro_usd = ahorro_tn * precio_gasoil

print('=== Impacto económico de la optimización IA ===')
print(f'Flota: {n_barcos} arrastreros de altura')
print(f'Mareas por año: {n_mareas_anio}')
print()
print(f'Ahorro promedio por marea: {ahorro_tn.mean():.1f} tn de gasoil')
print(f'  = USD {ahorro_usd.mean():,.0f} por marea')
print()
print(f'Ahorro anual por barco: {ahorro_tn.mean() * n_mareas_anio:.0f} tn')
print(f'  = USD {ahorro_usd.mean() * n_mareas_anio:,.0f} por barco/año')
print()
print(f'AHORRO TOTAL FLOTA (20 barcos, 1 año): USD {ahorro_usd.sum():,.0f}')
print(f'  ≈ USD {ahorro_usd.sum() / 1e6:.1f} millones/año')

# Visualización
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(n_barcos)
ax.bar(x - 0.2, consumo_sin_opt.mean(axis=1), 0.38,
       label='Sin optimización', color='#EF5350', alpha=0.85, edgecolor='white')
ax.bar(x + 0.2, consumo_con_opt.mean(axis=1), 0.38,
       label='Con optimización IA', color='#42A5F5', alpha=0.85, edgecolor='white')
ax.set_xlabel('Embarcación (1–20)')
ax.set_ylabel('Consumo promedio por marea (tn gasoil)')
ax.set_title('Impacto de la Optimización IA en el Consumo de Combustible\n'
             f'Flota de {n_barcos} arrastreros — Ahorro total: USD {ahorro_usd.sum():,.0f}/año',
             fontsize=12)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 8. Reflexión: ética y sostenibilidad

### El peligro de la optimización sin límites

Reducir el costo por marea puede tentarnos a hacer **más mareas** para maximizar la rentabilidad.
Pero el recurso pesquero tiene límites biológicos. La optimización IA debe operar **dentro**
del sistema de cuotas y vedas establecido por la autoridad pesquera, no para circunvalarlo.

### Optimización para la sostenibilidad

La misma lógica de optimización que reduce el consumo de combustible puede usarse para:
- **Minimizar el descarte:** rutas que eviten concentraciones de ejemplares juveniles
- **Respetar áreas de veda:** integrando zonas protegidas en la planificación de campaña
- **Reducir la huella de carbono:** menor consumo = menores emisiones de CO₂

### Los datos como bien común del sector

Los datos de flota, captura y variables ambientales son más valiosos si se comparten
que si cada empresa los guarda en silos. Iniciativas como Global Fishing Watch demuestran
que la transparencia de datos pesqueros beneficia a todo el sector.

---

## Para explorar más

- **Repo Ariel — ML operativo:** https://github.com/arielgiamportone/Machine_Learning_for_Sales_and_operations_Planning
- **PesquerosEnIA ML/DL:** https://github.com/PesquerosEnIA/ML_DL_FisheriesEngineers
- **K-Means clustering (scikit-learn):** https://scikit-learn.org/stable/modules/clustering.html
- **Optimización de rutas marítimas:** FAO Fisheries Circular No. 1119
- **Próxima clase (9):** Visualización y dashboards para la toma de decisiones